In [1]:
# Cell 1 — Mount Drive and extract DICOMs
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import tarfile

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS'
os.makedirs(DICOM_DIR, exist_ok=True)

print('Extracting DICOMs...')
with tarfile.open(f'{DATA_DIR}/fastmri_prostate_DICOMS_IDS_001_312.tar', 'r') as tar:
    tar.extractall(DICOM_DIR)

DICOM_DIR = '/content/DICOMS/DICOMS'
print(f'Patients: {len(os.listdir(DICOM_DIR))}')

Mounted at /content/drive
Extracting DICOMs...


/tmp/ipykernel_6748/3946954999.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DICOM_DIR)


Patients: 312


In [2]:
# Cell 2 — Installs
!pip install pydicom h5py optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.2 MB/s eta 0:00:00


In [3]:
# Cell 3 — Imports
import os
import numpy as np
import pandas as pd
import tarfile
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from PIL import Image
import torchvision.transforms as transforms
import optuna
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Cell 4 — Load labels and create splits
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks'
DICOM_DIR = '/content/DICOMS/DICOMS'

labels_tar = f'{DATA_DIR}/labels.tar'
with tarfile.open(labels_tar, 'r') as tar:
    f = tar.extractfile('labels/t2_slice_level_labels.csv')
    t2_labels = pd.read_csv(f)
    f = tar.extractfile('labels/volume_exam_labels.csv')
    volume_labels = pd.read_csv(f)

volume_labels['binary_label'] = (volume_labels['t2_volume_level'] >= 3).astype(int)
patient_split = t2_labels.drop_duplicates('fastmri_pt_id')[['fastmri_pt_id', 'data_split']]
df = patient_split.merge(volume_labels[['fastmri_pt_id', 'binary_label', 't2_volume_level']], on='fastmri_pt_id')

# Remove invalid patients with empty T2 folders
invalid_ids = [115, 258]
train_df = df[df['data_split'] == 'training'].reset_index(drop=True)
val_df   = df[df['data_split'] == 'validation'].reset_index(drop=True)
test_df  = df[df['data_split'] == 'test'].reset_index(drop=True)

train_df = train_df[~train_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
val_df   = val_df[~val_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)
test_df  = test_df[~test_df['fastmri_pt_id'].isin(invalid_ids)].reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')
print(df['binary_label'].value_counts())

Train: 217, Val: 47, Test: 46
binary_label
0    178
1    134
Name: count, dtype: int64


In [5]:
# Cell 5 — Dataset class (3x3 grid of middle 9 slices)
class ProstateT2Dataset(Dataset):
    def __init__(self, df, dicom_dir, augment=False):
        self.df = df.reset_index(drop=True)
        self.dicom_dir = dicom_dir

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])

        print(f'Dataset: {len(self.df)} patients')
        print(f'Labels: {self.df.binary_label.value_counts().to_dict()}')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pt_id = str(row.fastmri_pt_id).zfill(3)
        t2_path = os.path.join(self.dicom_dir, pt_id, 'AX_T2')

        slices = []
        for fname in os.listdir(t2_path):
            if fname.endswith('.dcm'):
                ds = pydicom.dcmread(os.path.join(t2_path, fname))
                slices.append((int(ds.InstanceNumber), ds.pixel_array.astype(np.float32)))

        slices.sort(key=lambda x: x[0])
        volume = np.stack([s[1] for s in slices])  # (30, 320, 320)

        # Per-volume min-max normalization
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)

        # Take middle 9 slices and tile into 3x3 grid
        mid = volume.shape[0] // 2
        selected = volume[mid-4:mid+5]  # 9 slices

        rows = []
        for i in range(3):
            row_img = np.concatenate([selected[i*3], selected[i*3+1], selected[i*3+2]], axis=1)
            rows.append(row_img)
        grid = np.concatenate(rows, axis=0)  # (960, 960)

        img = Image.fromarray((grid * 255).astype(np.uint8))
        img = img.convert('RGB')
        img = self.transform(img)

        label = torch.tensor(row.binary_label, dtype=torch.float32)
        return img, label

In [6]:
# Cell 6 — DataLoaders
train_dataset = ProstateT2Dataset(train_df, DICOM_DIR, augment=True)
val_dataset   = ProstateT2Dataset(val_df,   DICOM_DIR, augment=False)
test_dataset  = ProstateT2Dataset(test_df,  DICOM_DIR, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=4, pin_memory=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=0)

Dataset: 217 patients
Labels: {0: 129, 1: 88}
Dataset: 47 patients
Labels: {1: 26, 0: 21}
Dataset: 46 patients
Labels: {0: 26, 1: 20}


In [7]:
# Cell 7 — Test one sample
sample_img, sample_label = train_dataset[0]
print(f'Image shape: {sample_img.shape}')
print(f'Label: {sample_label}')
print(f'Min: {sample_img.min():.4f}, Max: {sample_img.max():.4f}')

Image shape: torch.Size([3, 224, 224])
Label: 0.0
Min: -2.1179, Max: 0.9145


In [8]:
# Cell 8 — Model definition and device
def get_model(dropout=0.5):
    model = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
    # Freeze first 4 stages
    for i, layer in enumerate(model.features):
        if i < 4:
            for param in layer.parameters():
                param.requires_grad = False
    # Replace classifier head
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 1)
    )
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [9]:
# Cell 9 — Train/Val functions
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).detach().cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.0
    return total_loss / len(loader), auc

In [10]:
# Cell 10 — Optuna hyperparameter search
n_neg = (train_df['binary_label'] == 0).sum()
n_pos = (train_df['binary_label'] == 1).sum()

def objective(trial):
    lr_backbone    = trial.suggest_float('lr_backbone',   1e-7, 1e-4, log=True)
    lr_head        = trial.suggest_float('lr_head',       1e-5, 1e-3, log=True)
    dropout        = trial.suggest_float('dropout',       0.1,  0.6)
    weight_decay   = trial.suggest_float('weight_decay',  1e-3, 0.1,  log=True)

    trial_model = get_model(dropout=dropout).to(device)

    backbone_params   = [p for p in trial_model.parameters()
                         if p.requires_grad and id(p) not in
                         set(id(p) for p in trial_model.classifier.parameters())]
    classifier_params = list(trial_model.classifier.parameters())

    optimizer = torch.optim.AdamW([
        {'params': backbone_params,   'lr': lr_backbone},
        {'params': classifier_params, 'lr': lr_head}
    ], weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

    pw        = torch.tensor([n_neg / n_pos]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_val_auc    = 0
    patience        = 5
    patience_counter = 0

    for epoch in range(20):
        train_epoch(trial_model, train_loader, optimizer, criterion, device)
        scheduler.step()
        _, val_auc = val_epoch(trial_model, val_loader, criterion, device)

        if val_auc > best_val_auc:
            best_val_auc     = val_auc
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

        trial.report(val_auc, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_auc


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
)
study.optimize(objective, n_trials=20, timeout=7200)  # 20 trials or 2 hours

print('\nBest trial:')
print(f'  Val AUC: {study.best_value:.4f}')
print(f'  Params: {study.best_params}')

[I 2026-05-18 00:11:48,368] A new study created in memory with name: no-name-857bc4f7-a63a-4c45-b5b9-1382d80fa3e9


Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:01<00:00, 189MB/s]
[I 2026-05-18 00:21:49,584] Trial 0 finished with value: 0.47619047619047616 and parameters: {'lr_backbone': 1.791043668775945e-06, 'lr_head': 0.0003599068188627684, 'dropout': 0.3220766789533318, 'weight_decay': 0.005124387937563316}. Best is trial 0 with value: 0.47619047619047616.
[I 2026-05-18 00:35:57,119] Trial 1 finished with value: 0.5201465201465202 and parameters: {'lr_backbone': 9.460086178343134e-07, 'lr_head': 6.530548346187231e-05, 'dropout': 0.11264850141929186, 'weight_decay': 0.028871605497825442}. Best is trial 1 with value: 0.5201465201465202.
[I 2026-05-18 00:44:15,390] Trial 2 finished with value: 0.5915750915750916 and parameters: {'lr_backbone': 3.1587374977160913e-06, 'lr_head': 1.99458033213913e-05, 'dropout': 0.1698845989773439, 'weight_decay': 0.01584508057556662}. Best is trial 2 with value: 0.5915750915750916.
[I 2026-05-18 00:49:11,682] Trial 3 pruned. 
[I 2026-05-18 01:04:59,938] Trial 4 finished with valu


Best trial:
  Val AUC: 0.6300
  Params: {'lr_backbone': 4.53108058309112e-07, 'lr_head': 4.4464106878560465e-05, 'dropout': 0.5965646668270674, 'weight_decay': 0.010429638770293254}


In [11]:
# Cell 11 — K-Fold Cross Validation with best hyperparameters
from sklearn.model_selection import StratifiedKFold

best = study.best_params
print(f'Running 5-fold CV with best params: {best}')

# Combine train + val for cross validation
combined_df = pd.concat([train_df, val_df]).reset_index(drop=True)
print(f'Combined train+val: {len(combined_df)} patients')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_val_aucs  = []
fold_test_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(combined_df, combined_df['binary_label'])):
    print(f'\n--- Fold {fold+1}/5 ---')

    fold_train_df = combined_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df   = combined_df.iloc[val_idx].reset_index(drop=True)

    # Datasets
    fold_train_dataset = ProstateT2Dataset(fold_train_df, DICOM_DIR, augment=True)
    fold_val_dataset   = ProstateT2Dataset(fold_val_df,   DICOM_DIR, augment=False)

    fold_train_loader = DataLoader(fold_train_dataset, batch_size=32, shuffle=True,
                                   num_workers=4, pin_memory=True, prefetch_factor=2)
    fold_val_loader   = DataLoader(fold_val_dataset,   batch_size=32, shuffle=False,
                                   num_workers=0)

    # Model
    fold_model = get_model(dropout=best['dropout']).to(device)

    backbone_params   = [p for p in fold_model.parameters()
                         if p.requires_grad and id(p) not in
                         set(id(p) for p in fold_model.classifier.parameters())]
    classifier_params = list(fold_model.classifier.parameters())

    fold_optimizer = torch.optim.AdamW([
        {'params': backbone_params,   'lr': best['lr_backbone']},
        {'params': classifier_params, 'lr': best['lr_head']}
    ], weight_decay=best['weight_decay'])

    fold_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(fold_optimizer, T_max=50)

    n_neg_fold = (fold_train_df['binary_label'] == 0).sum()
    n_pos_fold = (fold_train_df['binary_label'] == 1).sum()
    fold_pw    = torch.tensor([n_neg_fold / n_pos_fold]).to(device)
    fold_criterion = nn.BCEWithLogitsLoss(pos_weight=fold_pw)

    # Training
    best_fold_val_auc = 0
    patience_counter  = 0
    PATIENCE          = 10
    best_fold_path    = f'{DATA_DIR}/best_fold_{fold+1}.pth'

    for epoch in range(50):
        train_epoch(fold_model, fold_train_loader, fold_optimizer, fold_criterion, device)
        fold_scheduler.step()
        _, val_auc = val_epoch(fold_model, fold_val_loader, fold_criterion, device)

        if val_auc > best_fold_val_auc:
            best_fold_val_auc = val_auc
            patience_counter  = 0
            torch.save(fold_model.state_dict(), best_fold_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break

    # Load best model for this fold and evaluate on test
    fold_model.load_state_dict(torch.load(best_fold_path))
    _, test_auc = val_epoch(fold_model, test_loader, fold_criterion, device)

    print(f'Fold {fold+1} — Val AUC: {best_fold_val_auc:.4f} | Test AUC: {test_auc:.4f}')
    fold_val_aucs.append(best_fold_val_auc)
    fold_test_aucs.append(test_auc)

print(f'\n=== K-Fold Results ===')
print(f'Val AUC:  {np.mean(fold_val_aucs):.4f} ± {np.std(fold_val_aucs):.4f}')
print(f'Test AUC: {np.mean(fold_test_aucs):.4f} ± {np.std(fold_test_aucs):.4f}')

Running 5-fold CV with best params: {'lr_backbone': 4.53108058309112e-07, 'lr_head': 4.4464106878560465e-05, 'dropout': 0.5965646668270674, 'weight_decay': 0.010429638770293254}
Combined train+val: 264 patients

--- Fold 1/5 ---
Dataset: 211 patients
Labels: {0: 120, 1: 91}
Dataset: 53 patients
Labels: {0: 30, 1: 23}
Fold 1 — Val AUC: 0.6232 | Test AUC: 0.4808

--- Fold 2/5 ---
Dataset: 211 patients
Labels: {0: 120, 1: 91}
Dataset: 53 patients
Labels: {0: 30, 1: 23}
Fold 2 — Val AUC: 0.5710 | Test AUC: 0.4692

--- Fold 3/5 ---
Dataset: 211 patients
Labels: {0: 120, 1: 91}
Dataset: 53 patients
Labels: {0: 30, 1: 23}
Fold 3 — Val AUC: 0.5826 | Test AUC: 0.5192

--- Fold 4/5 ---
Dataset: 211 patients
Labels: {0: 120, 1: 91}
Dataset: 53 patients
Labels: {0: 30, 1: 23}
Fold 4 — Val AUC: 0.4565 | Test AUC: 0.5019

--- Fold 5/5 ---
Dataset: 212 patients
Labels: {0: 120, 1: 92}
Dataset: 52 patients
Labels: {0: 30, 1: 22}
Fold 5 — Val AUC: 0.4182 | Test AUC: 0.3692

=== K-Fold Results ===
Val A

In [12]:
# Cell 12 — Final test evaluation using best fold model
best_fold_idx = np.argmax(fold_val_aucs)
print(f'Best fold: {best_fold_idx+1} (Val AUC: {fold_val_aucs[best_fold_idx]:.4f})')

final_model = get_model(dropout=best['dropout']).to(device)
final_model.load_state_dict(torch.load(f'{DATA_DIR}/best_fold_{best_fold_idx+1}.pth'))

n_neg_all = (combined_df['binary_label'] == 0).sum()
n_pos_all = (combined_df['binary_label'] == 1).sum()
final_pw   = torch.tensor([n_neg_all / n_pos_all]).to(device)
final_criterion = nn.BCEWithLogitsLoss(pos_weight=final_pw)

_, final_test_auc = val_epoch(final_model, test_loader, final_criterion, device)
print(f'Final Test AUC: {final_test_auc:.4f}')

Best fold: 1 (Val AUC: 0.6232)
Final Test AUC: 0.4808


In [14]:
import json
import numpy as np

convnext_results = {
    'model': 'ConvNeXt-Small',
    'approach': '3x3 grid of middle 9 slices',
    'best_val_auc': float(np.mean(fold_val_aucs)),  # average across folds
    'fold_val_aucs': [float(x) for x in fold_val_aucs],
    'fold_test_aucs': [float(x) for x in fold_test_aucs],
    'test_auc': float(final_test_auc),
    'best_params': study.best_params,
    'n_train': len(train_df),
    'n_val': len(val_df),
    'n_test': len(test_df),
}

results_path = f'{DATA_DIR}/convnext_results.json'
with open(results_path, 'w') as f:
    json.dump(convnext_results, f, indent=2)

print('ConvNeXt results saved:')
print(json.dumps(convnext_results, indent=2))

ConvNeXt results saved:
{
  "model": "ConvNeXt-Small",
  "approach": "3x3 grid of middle 9 slices",
  "best_val_auc": 0.5303030303030303,
  "fold_val_aucs": [
    0.6231884057971014,
    0.5710144927536231,
    0.582608695652174,
    0.4565217391304348,
    0.4181818181818182
  ],
  "fold_test_aucs": [
    0.4807692307692308,
    0.46923076923076923,
    0.5192307692307692,
    0.5019230769230769,
    0.36923076923076925
  ],
  "test_auc": 0.4807692307692308,
  "best_params": {
    "lr_backbone": 4.53108058309112e-07,
    "lr_head": 4.4464106878560465e-05,
    "dropout": 0.5965646668270674,
    "weight_decay": 0.010429638770293254
  },
  "n_train": 217,
  "n_val": 47,
  "n_test": 46
}
